In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score



In [2]:
df = pd.read_csv("/Users/purikunsrinor/Project/Myself/kaggle practice/credit-risk-model/data/processed/cs-training_cleaned.csv", index_col=0)
df.shape

(150000, 12)

In [3]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

print(X.shape)
print(y.shape)

(150000, 11)
(150000,)


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Train default rate {y_train.mean():.3f}")
print(f"Val Default rate: {y_val.mean():.3f}")

Train: (120000, 11)
Val: (30000, 11)
Train default rate 0.067
Val Default rate: 0.067


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled =  scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict_proba(X_val_scaled)[:, 1]
lr_auc = roc_auc_score(y_val, lr_pred)
print(f"Logistic Regression AUC: {lr_auc:.4f}")

Logistic Regression AUC: 0.8117


In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict_proba(X_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_pred)
print(f"Random Forest AUC: {rf_auc:.4f}")

Random Forest AUC: 0.8407


In [8]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict_proba(X_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_pred)
print(f"XGBoost AUC: {xgb_auc:.4f}")

XGBoost AUC: 0.8597


In [9]:
results = {
    'Logistic Regression' : lr_auc,
    'Random Forest': rf_auc,
    'XGBoost': xgb_auc,
}

for model, auc in results.items():
    print(f"{model:25s} AUC: {auc:.4f}")

Logistic Regression       AUC: 0.8117
Random Forest             AUC: 0.8407
XGBoost                   AUC: 0.8597
